# Perbandingan koreksi GT dan matcher

Unggah `lensalabel_gt_matcher_comparison_bundle.zip`, lalu **Run all**.
40 cache OCR mentah tersedia dalam ZIP; tidak perlu foto atau OCR ulang.
CPU cukup: matching ini tidak memakai GPU. Split, KB, reading-order, closing-marker dan bobot tetap.

Empat kondisi: (a) GT lama + matcher lama, (b) GT baru + matcher lama,
(c) GT baru + multi-match/kepala bersama, serta ablasi multi-match tanpa kepala bersama.
Threshold 70/75/80/85/90/95. Prediksi (a) diverifikasi terhadap 240 baris hasil Colab sebelumnya.
Hasil tidak mengubah threshold produksi otomatis.


In [ ]:
from pathlib import Path
import os, json, hashlib, platform, subprocess, sys, zipfile
from datetime import datetime, timezone
from google.colab import files

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Unggah satu ZIP bundle eksperimen.'
root = Path('/content/lensalabel_gt_matcher_comparison')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    assert all((root / n).resolve().is_relative_to(root.resolve()) for n in z.namelist())
    z.extractall(root)
os.chdir(root)
manifest = json.loads(Path('bundle_manifest.json').read_text())
for name, digest in manifest['sha256'].items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, name
print('Bundle terverifikasi:', manifest['photos'], 'foto;', manifest['new_gt_ingredients'], 'anotasi GT baru')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz==3.14.5', 'SQLAlchemy>=2.0,<2.1', 'opencv-python-headless>=4.10', 'pytest>=8'], check=True)
env = dict(os.environ, PYTHONPATH=str(root / 'backend') + ':' + str(root))
subprocess.run([sys.executable, '-m', 'pytest', 'backend/tests', '-q'], env=env, check=True)


In [ ]:
output = Path('evaluation/results/gt_matcher_revision_colab')
output.mkdir(parents=True, exist_ok=True)
runtime = {'utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version, 'platform': platform.platform(), 'source': 'Colab', 'ocr_rerun': False}
(output / 'runtime.json').write_text(json.dumps(runtime, indent=2))
(output / 'pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
command = [sys.executable, '-u', 'evaluation/compare_gt_matchers.py',
    '--old-gt', 'dataset/ground_truth/correction_history/20261006_145456_transcript_mapping/calibration_ingredients_gt_final.json',
    '--new-gt', 'dataset/ground_truth/calibration_ingredients_gt_final.json',
    '--cache-dir', 'evaluation/results/colab_v4/ocr_cache',
    '--historical-detail', 'evaluation/results/colab_v4/grid_detail.csv',
    '--output-dir', str(output)]
with (output / 'run.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    if process.wait():
        raise RuntimeError('Eksperimen gagal; lihat run.log, jangan gunakan hasil parsial.')


In [ ]:
from IPython.display import Markdown, display
assert json.loads((output / 'run_manifest.json').read_text())['complete']
display(Markdown((output / 'comparison_report.md').read_text()))
archive = Path('/content/lensalabel_gt_matcher_comparison_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in output.rglob('*'):
        if path.is_file():
            z.write(path, path.relative_to(Path('evaluation/results')))
files.download(str(archive))


Ekstrak ZIP hasil ke `D:\Kulyeah\SKRIPSI\project\lensalabel\evaluation\results`.
Hasil akan berada di `gt_matcher_revision_colab/`. Hasil grid lama tetap tersimpan terpisah.
